# Traditional Materials Informatics – Photovoltaic Band Gap

Model comparison, **genetic-algorithm** descriptor selection (DEAP) and **applicability-domain** assessment (PCA, K-Means, distance), mirroring a classical QSAR/QSPR workflow. Full run: `pipeline/band_gap_ga_ad.py`.

In [ ]:
import sys, warnings, json
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT / "pipeline"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
sns.set_theme(style="whitegrid")
OUT = ROOT / "data/predictions/band_gap"
from IPython.display import Image, display
S = json.loads((OUT / 'band_gap_summary.json').read_text())
S['dataset']

In [ ]:
# 1. Model comparison on all descriptors
pd.DataFrame(S['model_comparison']).round(4)

In [ ]:
# 2. Genetic algorithm
ga = pd.read_csv(OUT / 'band_gap_ga_history.csv')
display(ga.round(4))
display(Image(str(OUT / 'band_gap_ga_evolution.png')))
{k: v for k, v in S['ga'].items() if k != 'selected_descriptors'}

In [ ]:
# Selected descriptors
pd.Series(S['ga']['selected_descriptors'])

In [ ]:
# 3. Test performance: all vs GA-selected descriptors
pd.DataFrame([S['test_all_descriptors'], S['test_ga_descriptors']]).round(4)

In [ ]:
# 4. Applicability domain
for f in ('band_gap_ad_pca.png', 'band_gap_ad_kmeans.png', 'band_gap_ad_distance.png'):
    display(Image(str(OUT / f)))
{k: v for k, v in S['applicability_domain'].items() if k != 'cluster_share'}

In [ ]:
# 5. Use the saved predictor
import joblib
from run_ssl_experiments import Preprocessor
import __main__
__main__.Preprocessor = Preprocessor
from featurize import featurize_formulas
pkg = joblib.load(ROOT / 'app/models/band_gap_model.pkl')
prep = joblib.load(ROOT / 'app/models/band_gap_scaler.pkl')
for f in ['CdTe', 'GaAs', 'Si', 'CsPbI3', 'Cu2ZnSnS4', 'Al2O3', 'Fe']:
    X = prep.transform(featurize_formulas(pd.Series([f]))[0])[:, pkg['selected_idx']]
    p = pkg['model'].predict_proba(X)[0, 1]
    d = pkg['nn'].kneighbors(X, n_neighbors=5)[0].mean()
    print(f"{f:<10} P(PV window)={p:.3f}  inside AD={d <= pkg['ad_threshold']}")